# Quickstart

*Author:* Hannu Parviainen<br>
*Last modified:* 30 October 2026

This quickstart models the transits of a real planet, HAT-P-2b, over a TESS-like observing window. It walks
through the steps you take with any planet: turning published parameters into the ones the model uses,
evaluating the model over a realistic time series, and checking the result.

It sticks to a single light curve on purpose. Multiple light curves, passbands, long-cadence supersampling,
transit timing variations, and parameter populations are shown in [Highlights](../highlights.rst).

In [ ]:
from matplotlib.pyplot import subplots, setp
from numpy import arange, radians, degrees, unique

from pytransit import RoadRunnerModel
from pytransit.orbits import as_from_rhop, i_from_ba, i_from_baew, d_from_pkaiews, epoch, fold, eclipse_phase

## The planet

HAT-P-2b ([Bakos et al. 2007](https://ui.adsabs.harvard.edu/abs/2007ApJ...670..826B/abstract)) is a hot
Jupiter of about nine Jupiter masses and 1.16 Jupiter radii, orbiting an F star every 5.63 days. For a planet
this close to its star the orbit is unusually eccentric, with $e \approx 0.52$, so it exercises the eccentric
side of the model as well as the basic one.

We take the parameters from [Pál et al. (2010)](https://ui.adsabs.harvard.edu/abs/2010MNRAS.401.2665P/abstract).

In [ ]:
t0 = 2454387.49375       # zero epoch [BJD]
p  = 5.6334729           # orbital period [d]
k  = 0.07227             # planet-star radius ratio
b  = 0.395               # impact parameter
e  = 0.5171              # eccentricity
w  = radians(185.22)     # argument of periastron [rad]

mstar, rstar = 1.36, 1.64   # stellar mass [M_Sun] and radius [R_Sun]

## From published parameters to model parameters

`evaluate` describes the orbit with the scaled semi-major axis $a/R_\star$ and the inclination $i$. Papers often
give the stellar density and the impact parameter instead, because those are what a transit light curve
constrains directly. The conversions are in `pytransit.orbits`:

- `as_from_rhop` gives $a/R_\star$ from the stellar density and the period (Kepler's third law), and
- `i_from_baew` gives the inclination from the impact parameter.

The eccentric version of the latter is needed because, on an eccentric orbit, the star-planet distance at
transit is not $a$, and the impact parameter becomes

$$b = \frac{a}{R_\star} \cos i \, \frac{1 - e^2}{1 + e \sin \omega}.$$

In [ ]:
rho = 1.41 * mstar / rstar**3      # stellar density [g/cm^3]; the Sun's mean density is 1.41 g/cm^3
a   = as_from_rhop(rho, p)
i   = i_from_baew(b, a, e, w)

print(f"ρ★ = {rho:.3f} g/cm³,  a/R★ = {a:.2f},  i = {degrees(i):.2f}°")

These agree with the $a/R_\star = 8.99$ and $i = 86.72°$ reported by Pál et al.

## Limb darkening

We use the power-2 law, a two-coefficient law, $I(\mu) = 1 - c(1 - \mu^\alpha)$, that follows stellar
atmosphere models more closely than the quadratic law.

The coefficients below are for HAT-P-2 in the TESS passband. They were computed with
[LDTk](https://github.com/hpparvi/ldtk) from the stellar parameters of Pál et al.,
$T_\mathrm{eff} = 6290 \pm 60$ K, $\log g = 4.16 \pm 0.03$, and $[\mathrm{Fe/H}] = 0.14 \pm 0.08$:

```python
from ldtk import LDPSetCreator
from ldtk.filters import tess

sc = LDPSetCreator(teff=(6290, 60), logg=(4.16, 0.03), z=(0.14, 0.08), filters=[tess])
ps = sc.create_profiles(nsamples=2000)
ldc, ldc_err = ps.coeffs_p2()
```

LDTk downloads the PHOENIX spectra it needs on first use, so the values are given directly here. See
[Limb darkening](../guide/limb_darkening.rst) for the other laws and for using LDTk profiles in the model
directly.

In [ ]:
ldc = [0.574, 0.715]    # power-2 c and α

## The observations

Next, one TESS-like sector: 27.4 days at 2-minute cadence, with a one-day gap in the middle where the
spacecraft downlinks its data. The times are BJDs, as in real data.

The zero epoch does not need to fall inside the observing window. The model counts the orbits from `t0` itself,
so a published ephemeris can be used as it is.

In [ ]:
cadence = 2 / 60 / 24                          # two minutes in days
time = arange(2459000.0, 2459027.4, cadence)
time = time[abs(time - 2459013.7) > 0.5]       # the mid-sector data gap

## The model

`RoadRunnerModel` is the recommended general-purpose model. It is fast and works with any radially symmetric
limb darkening profile; see [RoadRunner](../features/roadrunner.rst). Create it with a limb darkening law, give
it the times, and evaluate it.

In [ ]:
tm = RoadRunnerModel('power-2')
tm.set_data(time)
flux = tm.evaluate(k, ldc, t0, p, a, i, e, w)

The first `evaluate` call takes a few seconds while Numba compiles the model, and the calls after it are
fast. In practice you create the model and call `set_data` once, then call `evaluate` as many times as you
need; see the [User guide](../guide/index.rst).

In [ ]:
fig, ax = subplots(figsize=(13, 4))
ax.plot(time - 2459000, flux, '.', ms=1)
setp(ax, xlabel='Time - 2459000 [BJD]', ylabel='Normalised flux', xlim=(0, 27.4))
fig.tight_layout()

## A closer look

`epoch` gives the orbit number of each exposure counted from `t0`, and `fold` gives its time from the nearest
transit centre. Together they show which transits the sector caught, and stack them onto one another.
`d_from_pkaiews` gives the transit duration $T_{14}$ for the same orbit.

In [ ]:
ep    = epoch(time, t0, p)
phase = fold(time, p, t0)
t14   = d_from_pkaiews(p, k, a, i, e, w, 1)

print("Transit epochs in the sector:", unique(ep[flux < 1]))
print(f"T14 = {24 * t14:.2f} h")

In [ ]:
m = abs(phase) < 0.2

fig, ax = subplots(figsize=(13, 4))
ax.plot(24 * phase[m], flux[m], '.', ms=2)
for x in (-12 * t14, 12 * t14):
    ax.axvline(x, c='k', ls='--', lw=1)
setp(ax, xlabel='Time from transit centre [h]', ylabel='Normalised flux', xlim=(-4.8, 4.8))
fig.tight_layout()

All five transits fall on the same curve, and the dashed lines at $\pm T_{14}/2$ bracket it. The 4.30 h
duration agrees with the 4.29 h of Pál et al.

## The effect of eccentricity

To see what the eccentricity does, compare the model with a circular orbit that has the same $a/R_\star$ and
impact parameter. The circular orbit needs its own inclination from `i_from_ba`, and leaving `e` and `w` out of
`evaluate` makes the orbit circular.

In [ ]:
i_c    = i_from_ba(b, a)
flux_c = tm.evaluate(k, ldc, t0, p, a, i_c)
t14_c  = d_from_pkaiews(p, k, a, i_c, 0.0, 0.0, 1)

print(f"T14 = {24 * t14:.2f} h eccentric, {24 * t14_c:.2f} h circular")

fig, ax = subplots(figsize=(13, 4))
ax.plot(24 * phase[m], flux[m], '.', ms=2, label='Eccentric')
ax.plot(24 * phase[m], flux_c[m], '.', ms=2, label='Circular')
ax.legend()
setp(ax, xlabel='Time from transit centre [h]', ylabel='Normalised flux', xlim=(-4.8, 4.8))
fig.tight_layout()

HAT-P-2b transits on its way to periastron, when it is closer to its star and moving faster than it would on
a circular orbit. The transit is about half an hour shorter as a result, and its depth is unchanged.

Eccentricity also moves the secondary eclipse away from half an orbit after the transit. `eclipse_phase` gives
where it falls:

In [ ]:
print(f"Secondary eclipse at phase {eclipse_phase(p, i, e, w) / p:.3f}, "
      f"{eclipse_phase(p, i, e, w):.2f} d after the transit")

## Where next

- [Highlights](../highlights.rst): many light curves, passbands, and cadences in one model, a radius ratio per
  passband, transit timing variations, and parameter populations.
- [Data setup](../guide/data_setup.rst) and [Model evaluation](../guide/evaluation.rst): the details of
  `set_data` and `evaluate`.
- [Transit models](../models/index.rst): the other models, including oblate planets and gravity-darkened stars.

---

<center>©2026 Hannu Parviainen</center>